# 02 - Nsight Systems Trace-Guided Optimization Lab

This notebook is the hands-on lab. Each section targets one trace pattern, points to the smallest relevant source file, and compares the editable `problem` pipeline with the reference `solution` pipeline.

The intended loop for attendees is:

1. Profile the original problem pipeline.
2. Inspect the relevant module and the reference implementation.
3. Edit the problem module.
4. Re-profile and compare the region time, throughput, and Nsight Systems timeline.


## Setup

The profiling cells use CUDA and Nsight Systems when both are available. On CPU, the same cells still run the scripts and compare their printed region timers, but they skip `.nsys-rep` capture.


In [ ]:
from __future__ import annotations

from pathlib import Path
import difflib
import os
import re
import shutil
import subprocess
import sys
from typing import Sequence

from IPython.display import Markdown, display
import torch

ROOT = Path.cwd()
if not (ROOT / "profiling_workshop").exists() and (ROOT.parent / "profiling_workshop").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from profiling_workshop.common import cuda_is_available_quietly

TRACE_DIR = ROOT / "traces"
TRACE_DIR.mkdir(exist_ok=True)

PROBLEM_SCRIPT = ROOT / "scripts" / "run_problem_pipeline.py"
SOLUTION_SCRIPT = ROOT / "scripts" / "run_solution_pipeline.py"
PROBLEM_DIR = ROOT / "profiling_workshop" / "pipeline" / "problem"
SOLUTION_DIR = ROOT / "profiling_workshop" / "pipeline" / "solution"

PROFILE_DEVICE = "cuda" if cuda_is_available_quietly() else "cpu"
NSYS_AVAILABLE = shutil.which("nsys") is not None
RUN_NSYS = PROFILE_DEVICE == "cuda" and NSYS_AVAILABLE

print(f"Workshop root: {ROOT}")
print(f"Trace directory: {TRACE_DIR}")
print(f"Profile device: {PROFILE_DEVICE}")
print(f"nsys: {shutil.which('nsys') or 'not found'}")
if PROFILE_DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Workload Arguments

The CUDA workload uses fixed sample counts so changing batch size changes transfer frequency rather than total work. Adjust these values if your GPU is much smaller than an L40S/A100, or increase `--samples` for longer traces.


In [ ]:
if PROFILE_DEVICE == "cuda":
    COMMON_ARGS = [
        "--device", "cuda",
        "--samples", "8192",
        "--features", "2048",
        "--hidden", "4096",
        "--depth", "4",
        "--classes", "64",
        "--cpu-work", "2",
        "--log-every", "0",
    ]
    PROBLEM_ARGS = [
        *COMMON_ARGS,
        "--batch-size", "128",
        "--micro-batches", "16",
        "--num-workers", "0",
        "--prefetch-batches", "0",
        "--head", "broadcast-distance",
    ]
    SOLUTION_ARGS = [
        *COMMON_ARGS,
        "--batch-size", "1024",
        "--micro-batches", "1",
        "--num-workers", "4",
        "--prefetch-batches", "2",
        "--head", "matmul-distance",
    ]
else:
    COMMON_ARGS = [
        "--device", "cpu",
        "--samples", "256",
        "--features", "64",
        "--hidden", "128",
        "--depth", "2",
        "--classes", "8",
        "--cpu-work", "1",
        "--log-every", "0",
    ]
    PROBLEM_ARGS = [*COMMON_ARGS, "--batch-size", "32", "--micro-batches", "4", "--num-workers", "0"]
    SOLUTION_ARGS = [*COMMON_ARGS, "--batch-size", "64", "--micro-batches", "1", "--num-workers", "0", "--prefetch-batches", "0"]

PROBLEM_ARGS, SOLUTION_ARGS


## Helpers

`run_workload` captures the script output so we can compare the `RESULT` and `REGION` lines inside the notebook. `profile_workload` captures an Nsight Systems report with CUDA, NVTX, OS runtime, cuBLAS/cuDNN, CPU sampling, context switches, and CUDA memory usage.


In [ ]:
RESULT_RE = re.compile(
    r"RESULT pipeline=(?P<pipeline>\w+) device=(?P<device>\S+) samples=(?P<samples>\d+) "
    r"seconds=(?P<seconds>[0-9.]+) samples_per_second=(?P<sps>[0-9.]+) "
    r"loss=(?P<loss>[0-9.]+) accuracy=(?P<accuracy>[0-9.]+)"
)
REGION_RE = re.compile(
    r"REGION pipeline=(?P<pipeline>\w+) name=(?P<name>\S+) "
    r"seconds=(?P<seconds>[0-9.]+) percent_of_wall=(?P<percent>[0-9.]+)"
)


def run_command(cmd: Sequence[str], check: bool = True, capture: bool = False) -> subprocess.CompletedProcess[str]:
    cmd = [str(part) for part in cmd]
    print("\n$ " + " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=capture)


def parse_workload_output(output: str) -> dict[str, object]:
    result_match = RESULT_RE.search(output)
    if result_match is None:
        raise ValueError("The workload did not print a RESULT line.")
    result = result_match.groupdict()
    regions = {
        match.group("name"): {
            "seconds": float(match.group("seconds")),
            "percent": float(match.group("percent")),
        }
        for match in REGION_RE.finditer(output)
    }
    return {
        "pipeline": result["pipeline"],
        "samples": int(result["samples"]),
        "seconds": float(result["seconds"]),
        "samples_per_second": float(result["sps"]),
        "loss": float(result["loss"]),
        "accuracy": float(result["accuracy"]),
        "regions": regions,
        "output": output,
    }


def run_workload(label: str, script: Path, args: Sequence[str]) -> dict[str, object]:
    completed = run_command([sys.executable, script, *args], check=True, capture=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    parsed = parse_workload_output(completed.stdout)
    parsed["label"] = label
    return parsed


def comparison_table(problem: dict[str, object], solution: dict[str, object], region: str | None = None) -> None:
    rows = [
        "| metric | problem | solution | speedup |",
        "|---|---:|---:|---:|",
    ]
    p_sps = problem["samples_per_second"]
    s_sps = solution["samples_per_second"]
    rows.append(f"| throughput samples/s | {p_sps:,.1f} | {s_sps:,.1f} | {s_sps / max(p_sps, 1e-12):.2f}x |")
    rows.append(f"| wall seconds | {problem['seconds']:.3f} | {solution['seconds']:.3f} | {problem['seconds'] / max(solution['seconds'], 1e-12):.2f}x faster |")
    if region is not None:
        p_region = problem["regions"].get(region, {"seconds": 0.0})["seconds"]
        s_region = solution["regions"].get(region, {"seconds": 0.0})["seconds"]
        rows.append(f"| {region} seconds | {p_region:.6f} | {s_region:.6f} | {p_region / max(s_region, 1e-12):.2f}x faster |")
    display(Markdown("\n".join(rows)))


def show_source(path: Path, title: str | None = None) -> None:
    source = path.read_text().splitlines()
    numbered = "\n".join(f"{idx + 1:4d}  {line}" for idx, line in enumerate(source))
    heading = title or str(path.relative_to(ROOT))
    display(Markdown(f"### {heading}\n\n```python\n{numbered}\n```"))


def show_diff(problem_file: str) -> None:
    problem_path = PROBLEM_DIR / problem_file
    solution_path = SOLUTION_DIR / problem_file
    diff = difflib.unified_diff(
        problem_path.read_text().splitlines(),
        solution_path.read_text().splitlines(),
        fromfile=str(problem_path.relative_to(ROOT)),
        tofile=str(solution_path.relative_to(ROOT)),
        lineterm="",
    )
    display(Markdown("```diff\n" + "\n".join(diff) + "\n```"))


In [ ]:
NSYS_TRACE = os.environ.get("NSYS_TRACE", "cuda,nvtx,osrt,cublas,cudnn")
NSYS_EXTRA_ARGS = [
    f"--trace={NSYS_TRACE}",
    "--sample=process-tree",
    "--cpuctxsw=process-tree",
    "--backtrace=fp",
    "--cuda-memory-usage=true",
    "--stats=false",
]
NSYS_GPU_METRICS_DEVICE = os.environ.get("NSYS_GPU_METRICS_DEVICE", "auto")
NSYS_GPU_METRICS_FREQUENCY = os.environ.get("NSYS_GPU_METRICS_FREQUENCY", "10000")
_GPU_METRICS_DEVICE_CACHE: str | None | bool = False


def detect_gpu_metrics_device() -> str | None:
    global _GPU_METRICS_DEVICE_CACHE
    if _GPU_METRICS_DEVICE_CACHE is not False:
        return _GPU_METRICS_DEVICE_CACHE
    if not RUN_NSYS:
        _GPU_METRICS_DEVICE_CACHE = None
        return None
    requested = NSYS_GPU_METRICS_DEVICE.lower()
    if requested in {"", "none", "off", "false"}:
        _GPU_METRICS_DEVICE_CACHE = None
        return None
    if requested != "auto":
        _GPU_METRICS_DEVICE_CACHE = NSYS_GPU_METRICS_DEVICE
        return NSYS_GPU_METRICS_DEVICE

    completed = subprocess.run(
        ["nsys", "profile", "--gpu-metrics-device=help", "--duration=1", "--trace=none", "true"],
        text=True,
        capture_output=True,
        check=False,
    )
    help_text = completed.stdout + completed.stderr
    for line in help_text.splitlines():
        text = line.strip()
        if not text or text.lower().startswith("possible"):
            continue
        if "not found" in text.lower() or "not available" in text.lower():
            continue
        if text == "all" or text.startswith("all "):
            _GPU_METRICS_DEVICE_CACHE = "all"
            return "all"
        match = re.match(r"^(\d+)(?:\s|:|$)", text)
        if match:
            _GPU_METRICS_DEVICE_CACHE = match.group(1)
            return _GPU_METRICS_DEVICE_CACHE

    print("GPU HW metrics were not enabled because Nsight did not report a supported metrics device.")
    _GPU_METRICS_DEVICE_CACHE = None
    return None


def remove_stale_outputs(report_base: Path) -> None:
    for suffix in [".nsys-rep", ".sqlite", ".qdstrm"]:
        candidate = report_base.with_suffix(suffix)
        if candidate.exists():
            candidate.unlink()


def resolve_report(report_base: Path) -> Path | None:
    report = report_base.with_suffix(".nsys-rep")
    if report.exists():
        return report
    sqlite = report_base.with_suffix(".sqlite")
    if sqlite.exists():
        return sqlite
    return None


def profile_workload(name: str, script: Path, args: Sequence[str]) -> Path | None:
    if not RUN_NSYS:
        print("Skipping Nsight Systems capture because CUDA or nsys is unavailable.")
        return None
    report_base = TRACE_DIR / name
    remove_stale_outputs(report_base)
    cmd = [
        "nsys",
        "profile",
        "--force-overwrite=true",
        *NSYS_EXTRA_ARGS,
    ]
    metrics_device = detect_gpu_metrics_device()
    if metrics_device is not None:
        cmd.extend([
            f"--gpu-metrics-device={metrics_device}",
            f"--gpu-metrics-frequency={NSYS_GPU_METRICS_FREQUENCY}",
        ])
        print(f"GPU metrics enabled for device setting: {metrics_device}")
    cmd.extend([
        "-o", str(report_base),
        sys.executable,
        str(script),
        *args,
    ])
    completed = run_command(cmd, check=False, capture=False)
    report = resolve_report(report_base)
    if report is not None:
        print(f"Nsight report: {report}")
    elif completed.returncode != 0:
        print(f"nsys exited with return code {completed.returncode}")
    return report


def nsys_stats(report: Path | None) -> None:
    if report is None:
        print("No Nsight Systems report to summarize.")
        return
    if shutil.which("nsys") is None:
        print("nsys was not found on PATH.")
        return
    cmd = [
        "nsys",
        "stats",
        "--force-export=true",
        "--report", "cuda_api_sum,cuda_gpu_mem_time_sum,cuda_gpu_mem_size_sum,cuda_gpu_kern_sum,nvtx_sum,nvtx_kern_sum,osrt_sum",
        "--format", "table",
        str(report),
    ]
    run_command(cmd, check=False, capture=False)


def profile_pair(issue_name: str) -> tuple[Path | None, Path | None]:
    problem_report = profile_workload(f"{issue_name}_problem", PROBLEM_SCRIPT, PROBLEM_ARGS)
    solution_report = profile_workload(f"{issue_name}_solution", SOLUTION_SCRIPT, SOLUTION_ARGS)
    return problem_report, solution_report


## Original Outputs

Run both pipelines once before making edits. Keep these numbers as the starting point for the lab.


In [ ]:
original_problem = run_workload("original problem", PROBLEM_SCRIPT, PROBLEM_ARGS)
reference_solution = run_workload("reference solution", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(original_problem, reference_solution)


## 1. Remove Extra Synchronization

Target file: `profiling_workshop/pipeline/problem/synchronization.py`

Look for CUDA API rows such as `cudaDeviceSynchronize`, D2H copies attached to metric logging, and NVTX ranges beginning with `issue_1_`.


In [ ]:
show_source(PROBLEM_DIR / "synchronization.py", "Editable problem synchronization module")
show_diff("synchronization.py")


In [ ]:
sync_problem = run_workload("problem after sync edits", PROBLEM_SCRIPT, PROBLEM_ARGS)
sync_solution = run_workload("solution sync reference", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(sync_problem, sync_solution, "issue_1_synchronization")


In [ ]:
sync_problem_report, sync_solution_report = profile_pair("issue_1_sync")
nsys_stats(sync_problem_report)
nsys_stats(sync_solution_report)


## 2. Increase Useful Work per Launch

Target file: `profiling_workshop/pipeline/problem/short_kernels.py`

The original path processes each batch as many tiny optimizer steps. In Nsight Systems, the `issue_2_short_lived_microbatch_train_step` range should contain many kernel launches whose launch overhead is large relative to execution time.


In [ ]:
show_source(PROBLEM_DIR / "short_kernels.py", "Editable problem short-kernel module")
show_diff("short_kernels.py")


In [ ]:
kernel_problem = run_workload("problem after kernel edits", PROBLEM_SCRIPT, PROBLEM_ARGS)
kernel_solution = run_workload("solution kernel reference", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(kernel_problem, kernel_solution, "issue_2_short_lived_kernels")


In [ ]:
kernel_problem_report, kernel_solution_report = profile_pair("issue_2_kernels")
nsys_stats(kernel_problem_report)
nsys_stats(kernel_solution_report)


## 3. Overlap CPU and GPU Work

Target file: `profiling_workshop/pipeline/problem/handoffs.py`

The original code prepares CPU features in the main training loop. The reference path moves that work into the DataLoader side of the pipeline, allowing workers to prepare future batches while the GPU trains on the current one.


In [ ]:
show_source(PROBLEM_DIR / "handoffs.py", "Editable problem CPU/GPU handoff module")
show_diff("handoffs.py")


In [ ]:
handoff_problem = run_workload("problem after handoff edits", PROBLEM_SCRIPT, PROBLEM_ARGS)
handoff_solution = run_workload("solution handoff reference", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(handoff_problem, handoff_solution, "issue_3_cpu_gpu_handoff")


In [ ]:
handoff_problem_report, handoff_solution_report = profile_pair("issue_3_handoff")
nsys_stats(handoff_problem_report)
nsys_stats(handoff_solution_report)


## 4. Reduce Host/Device IO Frequency

Target file: `profiling_workshop/pipeline/problem/batching.py`

The original path uses small batches, pageable host memory, and blocking H2D copies. The reference path uses larger batches, pinned DataLoader memory, and CUDA-stream prefetching so the next copy can be queued before the current batch finishes computing.


In [ ]:
show_source(PROBLEM_DIR / "batching.py", "Editable problem batching and IO module")
show_diff("batching.py")


In [ ]:
io_problem = run_workload("problem after IO edits", PROBLEM_SCRIPT, PROBLEM_ARGS)
io_solution = run_workload("solution IO reference", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(io_problem, io_solution, "issue_4_batch_io")


In [ ]:
io_problem_report, io_solution_report = profile_pair("issue_4_io")
nsys_stats(io_problem_report)
nsys_stats(io_solution_report)


## Final Check

After all edits, run the problem pipeline one more time and compare it against the reference solution. A successful workshop fix should not need to match the reference exactly, but the timeline should show fewer synchronization barriers, fewer tiny launch clusters, better CPU/GPU overlap, and less copy overhead per sample.


In [ ]:
final_problem = run_workload("final problem", PROBLEM_SCRIPT, PROBLEM_ARGS)
comparison_table(final_problem, reference_solution)
